# BlackMarblePy: Exercise 1 [Gas Flaring in Nigeria]

This exercise examines nighttime lights in and outside gas flare locations in Nigeria. There are two parts:

__1. Calculate NTL at the ADM0 level, including and excluding gas flares.__

__2. Calculate NTL within each gas flare buffer.__

## Setup

Run these setup cells first. They load packages, find the repository root, read your NASA Earthdata bearer token, and load Nigeria boundaries and gas flare locations.

### Import Packages

In [10]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import dotenv_values

from blackmarble import Product, extract

### Set Project Paths

In [11]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "spatial-blackmarble-training").exists():
            return path
    return start


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "ntl_blackmarble" / "nigeria" / "raw" # change this to your directory
RAW_DIR.mkdir(parents=True, exist_ok=True)

### Load Black Marble Token

In [12]:
secrets_path = REPO_ROOT / ".config" / "ntl-training" / "secrets.env"
secrets = dotenv_values(secrets_path)
blackmarble_token = secrets.get("BLACKMARBLE_TOKEN", "").strip()

### Load Nigeria Boundary

In [13]:
nga0 = gpd.read_file(DATA_DIR / "gadm" / "nga_adm0.geojson")
nga0["geometry"] = nga0.geometry.make_valid()
nga0 = nga0.to_crs("EPSG:4326")

nga0.head()

,COUNTRY,geometry
0,Nigeria,"MULTIPOLYGON (((6.43875 4.39375, 6.43904 4.393..."


### Load Gas Flare Locations

In [14]:
gf = pd.read_csv(DATA_DIR / "gas_flaring" / "finaldata" / "gas_flaring_nga.csv")

gf_points = gpd.GeoDataFrame(
    gf,
    geometry=gpd.points_from_xy(gf.longitude, gf.latitude),
    crs="EPSG:4326",
)

gf_points.head()

,country,latitude,longitude,bcm,m_mscfd,year,field_type,field_name,field_operator,location,flare_level,flaring_vol_million_m3,geometry
0,Nigeria,5.619424,6.815036,0.109629,10.606928,2012,OIL,Izombe,Addax,ONSHORE,Medium,109.629477,POINT (6.81504 5.61942)
1,Nigeria,5.673479,5.274604,0.059017,5.710021,2012,OIL,Abiteye,Chevron,ONSHORE,Medium,59.016766,POINT (5.2746 5.67348)
2,Nigeria,5.860803,5.133084,0.111757,10.812796,2012,OIL,Benin River,Chevron,ONSHORE,Medium,111.757254,POINT (5.13308 5.8608)
3,Nigeria,5.817179,5.200836,0.126999,12.287433,2012,OIL,Dibi,Chevron,ONSHORE,Medium,126.998587,POINT (5.20084 5.81718)
4,Nigeria,5.783992,5.428424,0.064862,6.275607,2012,OIL,Makaraba,Chevron,ONSHORE,Medium,64.862461,POINT (5.42842 5.78399)


### Create Gas Flare Buffers

In [15]:
# EPSG:32632 is UTM Zone 32N, a useful metric CRS for much of Nigeria.
gf_5km = gf_points.copy()
gf_5km["geometry"] = gf_5km.to_crs(epsg=32632).buffer(5000).to_crs("EPSG:4326")
gf_5km_union = gf_5km.union_all()

gf_5km.head()

,country,latitude,longitude,bcm,m_mscfd,year,field_type,field_name,field_operator,location,flare_level,flaring_vol_million_m3,geometry
0,Nigeria,5.619424,6.815036,0.109629,10.606928,2012,OIL,Izombe,Addax,ONSHORE,Medium,109.629477,"POLYGON ((6.86015 5.61959, 6.85995 5.61516, 6...."
1,Nigeria,5.673479,5.274604,0.059017,5.710021,2012,OIL,Abiteye,Chevron,ONSHORE,Medium,59.016766,"POLYGON ((5.31966 5.67377, 5.31947 5.66934, 5...."
2,Nigeria,5.860803,5.133084,0.111757,10.812796,2012,OIL,Benin River,Chevron,ONSHORE,Medium,111.757254,"POLYGON ((5.17815 5.86111, 5.17796 5.85669, 5...."
3,Nigeria,5.817179,5.200836,0.126999,12.287433,2012,OIL,Dibi,Chevron,ONSHORE,Medium,126.998587,"POLYGON ((5.2459 5.81748, 5.24571 5.81306, 5.2..."
4,Nigeria,5.783992,5.428424,0.064862,6.275607,2012,OIL,Makaraba,Chevron,ONSHORE,Medium,64.862461,"POLYGON ((5.4735 5.78427, 5.47331 5.77985, 5.4..."


In [16]:
gf_5km.explore()

### Set Extraction Parameters

In [17]:
date_range_annual = pd.date_range("2023-01-01", "2024-01-01", freq="YS")
extract_kwargs = dict(
    product_id=Product.VNP46A4,
    date_range=date_range_annual,
    token=blackmarble_token,
    output_directory=None,
    output_skip_if_exists=True,
)

## Part 1: Nighttime lights at ADM0 level

Determine the sum of nighttime lights annually from 2012 to 2024 across Nigeria. Make a figure that shows trends in NTL:

* Across Nigeria
* Across Nigeria, only considering gas flaring locations
* Across Nigeria, excluding gas flaring locations

Decisions to make:

* What buffer should you use for excluding gas flaring locations?
* How should you include or exclude gas flaring locations?

In [ ]:
# Starter code

# Create geometries for Nigeria inside and outside the gas flare buffers.
nga0_gf = nga0.copy()
nga0_gf["geometry"] = ...

nga0_nogf = nga0.copy()
nga0_nogf["geometry"] = ...

# Extract annual NTL sums for each geometry.
ntl_total = extract.bm_extract(nga0, **extract_kwargs)
ntl_gf = ...
ntl_nogf = ...

# Combine and plot the trends.

### Solution

In [ ]:
# Make a copy of the Nigeria ADM0 boundary so we can create a gas-flaring-only geometry.
nga0_gf = nga0.copy()

# Keep only the parts of Nigeria that overlap the 5 km gas flare buffers, then repair any invalid geometry.
nga0_gf["geometry"] = nga0_gf.geometry.intersection(gf_5km_union).make_valid()

# Drop empty or invalid geometries created by the intersection and reset row numbers.
nga0_gf = nga0_gf[~nga0_gf.is_empty & nga0_gf.geometry.is_valid].reset_index(drop=True)

# Make another copy of the Nigeria boundary for the non-gas-flaring area.
nga0_nogf = nga0.copy()

# Remove the 5 km gas flare buffer areas from Nigeria, then repair any invalid geometry.
nga0_nogf["geometry"] = nga0_nogf.geometry.difference(gf_5km_union).make_valid()

# Drop empty or invalid geometries created by the difference operation and reset row numbers.
nga0_nogf = nga0_nogf[~nga0_nogf.is_empty & nga0_nogf.geometry.is_valid].reset_index(drop=True)

# Extract annual nighttime lights for all of Nigeria and label this result as the total series.
ntl_total = extract.bm_extract(nga0, **extract_kwargs).assign(series="NTL: Total")

# Extract annual nighttime lights only inside the gas flare buffer areas.
ntl_gf = extract.bm_extract(nga0_gf, **extract_kwargs).assign(series="NTL: Gas Flaring")

# Extract annual nighttime lights for Nigeria after excluding the gas flare buffer areas.
ntl_nogf = extract.bm_extract(nga0_nogf, **extract_kwargs).assign(series="NTL: Exclude Gas Flaring")

# Stack the three extraction results into one table for plotting.
ntl_trends = pd.concat([ntl_total, ntl_gf, ntl_nogf], ignore_index=True)

# Convert the date column from text to datetime so we can work with calendar fields.
ntl_trends["date"] = pd.to_datetime(ntl_trends["date"])

# Pull out the year from each date for the x-axis.
ntl_trends["year"] = ntl_trends["date"].dt.year

# Create a figure and axis for the line chart.
fig, ax = plt.subplots(figsize=(9, 5))

# Plot one line for each series: total, gas-flaring-only, and excluding gas flares.
for series, group in ntl_trends.groupby("series"):
    ax.plot(group["year"], group["ntl_sum"], linewidth=2, label=series)

# Remove the x-axis label because the years are self-explanatory.
ax.set_xlabel(None)

# Label the y-axis as radiance, the unit represented by the nighttime lights sum.
ax.set_ylabel("Radiance")

# Add a legend and remove its border for a cleaner plot.
ax.legend(frameon=False)

# Add a descriptive chart title.
ax.set_title("Annual nighttime lights in Nigeria")

# Display the plot.
plt.show()

['h18v07', 'h18v08', 'h19v07', 'h19v08']
['h18v07', 'h18v08', 'h19v07', 'h19v08']
['https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/archives/VNP46A4.A2024001.h18v07.002.2025162031135.h5', 'https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/archives/VNP46A4.A2023001.h18v07.002.2025161113010.h5', 'https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/archives/VNP46A4.A2023001.h18v08.002.2025161113203.h5', 'https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/archives/VNP46A4.A2024001.h18v08.002.2025162031854.h5', 'https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/archives/VNP46A4.A2023001.h19v07.002.2025161112717.h5', 'https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/archives/VNP46A4.A2024001.h19v07.002.2025162121907.h5', 'https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/archives/VNP46A4.A2024001.h19v08.002.2025162033234.h5', 'https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/archives/VNP46A4.A2023001.h19v08.002.2025161112653.h5']


  0%|          | 0.00/73.7M [00:00<?, ?B/s]

  0%|          | 0.00/72.4M [00:00<?, ?B/s]

  0%|          | 0.00/85.5M [00:00<?, ?B/s]

  0%|          | 0.00/86.0M [00:00<?, ?B/s]

  0%|          | 0.00/71.3M [00:00<?, ?B/s]

  0%|          | 0.00/73.9M [00:00<?, ?B/s]

## Part 2: Nighttime lights in gas flaring locations

Calculate the sum of NTL around each gas flaring location in 2024. Using this data:

1. Make a figure showing the distribution of NTL across gas flaring locations.
2. Map gas flaring buffers, coloring each buffer by the sum of NTL.

In [ ]:
# Starter code

ntl_gf_2024 = extract.bm_extract(
    ...,  # region of interest
    product_id=...,  # product
    date_range=...,  # date
    token=blackmarble_token,
    output_directory=str(RAW_DIR),
    output_skip_if_exists=True,
)

# Histogram

# Map

### Solution

In [ ]:
ntl_gf_2024 = extract.bm_extract(
    gf_5km,
    product_id=Product.VNP46A4,
    date_range=pd.date_range("2024-01-01", "2024-01-01", freq="YS"),
    token=blackmarble_token,
    output_directory=str(RAW_DIR),
    output_skip_if_exists=True,
)

gf_5km_ntl = gf_5km.copy()
gf_5km_ntl["ntl_sum"] = ntl_gf_2024["ntl_sum"].to_numpy()

fig, ax = plt.subplots(figsize=(8, 4))
gf_5km_ntl["ntl_sum"].plot.hist(ax=ax, bins=30, color="darkorange", edgecolor="black")
ax.set_xlabel("Radiance")
ax.set_ylabel("N Gas Flaring Locations")
ax.set_title("Distribution of NTL across gas flaring locations")
plt.show()

fig, ax = plt.subplots(figsize=(8, 8))
gf_5km_ntl.plot(column="ntl_sum", ax=ax, legend=True, cmap="magma")
nga0.boundary.plot(ax=ax, color="black", linewidth=0.7)
ax.set_title("NTL around gas flaring locations, Nigeria, 2024")
ax.set_axis_off()
plt.show()